In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-01-16 09:07:18 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



## Introducción

En la tabla de vinculaciones `resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf` se obtiene las vinculaciones o afiliaciones.

Las cifras de negocios no tienen en cuenta que un cliente puede generar varios registros en diferentes meses y años [el cliente va y vuelve, adiciona nueva franquicia], no se parte de la primera vinculación.

En el archivo `uso_adquirencia.ipynb` se tiene en cuenta el enfoque de la primera vinculación o afiliación.


## Construcción históricos

Se calcula la métrica uso para tres escenarios

- Todos los clientes
- Clientes nuevos
- Clientes viejos

Se dice que un cliente [todos, nuevos o viejos] tiene uso cuando realiza al menos una (1) transaccion en el año corriente. 

Importante tener en cuenta para clientes nuevos. Un cliente que vinculó adquirencia en el año 2024 [nuevo en 2024] y realizó un transacción en el mismo año suma a la métrica, en cambio si ese mismo cliente realiza una transacción en el año 2025 y siguiente no sumará a la métrica.

In [2]:
# Tabla que almacenará comercios con trxs por periodo
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup  (
                codigo_unico DOUBLE,
                periodo DOUBLE,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2),
                tipo_cliente STRING,
                producto STRING
                )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup;"""
helper.ejecutar_consulta(sql_compute)

2026-01-16 09:12:28 - [INFO] - Transcurrido: 1768572748, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ...i_vinculaciones_con_trxs_hist_con_dup   finalizado   09:12:36 AM     00:00.6 
------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
  i   tipo                   nombre                     estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 2/2 CREATE ...i_vinculaciones_con_trxs_hist_con_dup   finalizado   09:12:37 AM     00:00.3 
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------

In [3]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2023-01-01' # MODIFICAR. INICIO DE UN MES. EN ACTUALIZACIÓN USAR EL SIGUIENTE MES DESPUÉS DEL ÚLTIMO ALMACENADO
fecha_final = '2025-11-30' # MODIFICAR. FIN DE UN MES. EN ACTUALIZACIÓN USAR EL MES RECIENTE CON TRANSACCIONES COMPLETAS
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)
df_config['periodo_viejos'] = df_config['fechas'].apply(lambda x: str(x.year - 1) + '12')
# df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
# df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
# df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
# df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
# df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_sgte_mes,year_sgte_mes,periodo_viejos
0,2023-01-31,2023,1,31,202301,202301,202302,2023,202212
1,2023-02-28,2023,2,28,202302,202301,202303,2023,202212
2,2023-03-31,2023,3,31,202303,202301,202304,2023,202212
3,2023-04-30,2023,4,30,202304,202301,202305,2023,202212
4,2023-05-31,2023,5,31,202305,202301,202306,2023,202212
5,2023-06-30,2023,6,30,202306,202301,202307,2023,202212
6,2023-07-31,2023,7,31,202307,202301,202308,2023,202212
7,2023-08-31,2023,8,31,202308,202301,202309,2023,202212
8,2023-09-30,2023,9,30,202309,202301,202310,2023,202212
9,2023-10-31,2023,10,31,202310,202301,202311,2023,202212


In [ ]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')

    print('Viejos')
    print('') 
    print('Obteniendo vinculaciones acumuladas al último mes dsel año anterior correspondiente al Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
    SELECT codigo_unico,
          extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'year')*100 + extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'month') AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
      WHERE YEAR <= """ + str(row.year_sgte_mes) + """
      AND MONTH BETWEEN 1 AND 12
      AND DAY BETWEEN 1 AND 31
      ), viejos AS (
      SELECT codigo_unico, periodo
      FROM vinc
      WHERE periodo <= """ + row.periodo_viejos + """
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM viejos;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'viejos' as tipo_cliente,
         'adqui' as producto
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')


    print('Nuevos')
    print('') 
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
    SELECT codigo_unico,
          extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'year')*100 + extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'month') AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
      WHERE YEAR BETWEEN """ + str(row.year) + """ AND """ + str(row.year_sgte_mes) + """
      AND MONTH BETWEEN 1 AND 12
      AND DAY BETWEEN 1 AND 31
    ), nuevos AS (
    SELECT codigo_unico, periodo
      FROM vinc
      WHERE periodo BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM nuevos;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'nuevos' as tipo_cliente,
         'adqui' as producto
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    print('Todos')
    print('') 
    print('Obteniendo vinculaciones acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
    SELECT codigo_unico,
          extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'year')*100 + extract(to_timestamp(concat(cast(YEAR AS string), '-', cast(MONTH AS string), '-01'), 'yyyy-M-dd') - INTERVAL 1 MONTH, 'month') AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
      WHERE YEAR <= """ + str(row.year_sgte_mes) + """
      AND MONTH BETWEEN 1 AND 12
      AND DAY BETWEEN 1 AND 31
    ), todos AS (
    SELECT codigo_unico, periodo
      FROM vinc
      WHERE periodo <= """ + str(row.periodo) + """
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM todos;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'todos' as tipo_cliente,
         'adqui' as producto
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    

2026-01-16 11:08:45 - [INFO] - Transcurrido: 6967, Tiempo de Refresco = 1000


##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2023 - 1

Viejos

Obteniendo vinculaciones acumuladas al último mes del año anterior correspondiente al Mes de análisis

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 4/4    DROP ...so.mdo_adquirencia_vinculaciones_temp   finalizado   11:08:46 AM     00:00.4 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 5/5  CREATE ...so.mdo_adquirencia

In [7]:
# Obtener uso por mes
# Número de Vinculaciones
sql = """
SELECT producto,
       tipo_cliente,
       periodo,
       count(*) AS num_vinc_uso_cumsum_ym
FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist_con_dup
GROUP BY 1,
         2,
         3
ORDER BY periodo DESC, tipo_cliente, producto;
"""
df_prueba = helper.obtener_dataframe(sql)

2026-01-16 13:49:05 - [INFO] - Transcurrido: 4215, Tiempo de Refresco = 1000


---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 740/740 DATAFRAME                                           descargando   01:49:05 PM             

2026-01-16 13:49:17 - [INFO] - 140 filas, 4 columnas, 00:12.5 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 740/740 DATAFRAME                                            finalizado   01:49:05 PM     00:12.7 
---------------------------------------------------------------------------------------------------


In [8]:
df_prueba.head(20)

,producto,tipo_cliente,periodo,num_vinc_uso_cumsum_ym
0,adqui,nuevos,202511.0,25097
1,adqui,todos,202511.0,111140
2,wompi,todos,202511.0,30957
3,adqui,viejos,202511.0,86043
4,adqui,nuevos,202510.0,22951
5,adqui,todos,202510.0,108753
6,wompi,todos,202510.0,28158
7,adqui,viejos,202510.0,85802
8,adqui,nuevos,202509.0,20486
9,adqui,todos,202509.0,106000
